In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import (mean_absolute_percentage_error, mean_absolute_error,
                             root_mean_squared_error, r2_score)

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train_log = pd.read_csv("../data/processed/y_train_log.csv").squeeze("columns")
grup_train = pd.read_csv("../data/processed/grup_train.csv").squeeze("columns")

print(X_train.shape, y_train_log.shape, grup_train.shape)
print("jumlah grup:", grup_train.nunique())

(2533, 111) (2533,) (2533,)
jumlah grup: 2291


In [3]:
gkf = GroupKFold(n_splits=5)

def evaluasi_cv(model, X, y_log, grup):
    hasil = []
    for idx_latih, idx_uji in gkf.split(X, y_log, groups=grup):
        m = clone(model)
        m.fit(X.iloc[idx_latih], y_log.iloc[idx_latih])

        pred = np.expm1(m.predict(X.iloc[idx_uji]))   # log -> rupiah
        asli = np.expm1(y_log.iloc[idx_uji])

        hasil.append({
            "MAPE (%)":    mean_absolute_percentage_error(asli, pred) * 100,
            "MAE (juta)":  mean_absolute_error(asli, pred) / 1e6,
            "RMSE (juta)": root_mean_squared_error(asli, pred) / 1e6,
            "R2":          r2_score(asli, pred),
        })
    return pd.DataFrame(hasil).mean()

In [4]:
MIRING = ["land_size_m2", "building_size_m2", "carports", "garages",
          "maid_bedrooms", "maid_bathrooms", "building_age", "listrik"]

log_fitur = ColumnTransformer(
    [("log", FunctionTransformer(np.log1p), MIRING)],
    remainder="passthrough",
)

hasil = {}
hasil["Baseline: tebak median"] = evaluasi_cv(
    DummyRegressor(strategy="median"), X_train, y_train_log, grup_train)
hasil["Ridge (fitur mentah)"] = evaluasi_cv(
    make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X_train, y_train_log, grup_train)
hasil["Ridge (fitur miring di-log)"] = evaluasi_cv(
    make_pipeline(log_fitur, StandardScaler(), Ridge(alpha=1.0)), X_train, y_train_log, grup_train)

pd.DataFrame(hasil).T.round(2)

,MAPE (%),MAE (juta),RMSE (juta),R2
Baseline: tebak median,91.30,3230.62,10361.29,-0.07
Ridge (fitur mentah),47.21,13323.39,251398.33,-1663.83
Ridge (fitur miring di-log),24.60,1212.76,5339.61,0.74


In [5]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

model_pohon = {
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost":       XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM":      LGBMRegressor(random_state=42, verbose=-1),
}

for nama, model in model_pohon.items():
    hasil[nama] = evaluasi_cv(model, X_train, y_train_log, grup_train)

pd.DataFrame(hasil).T.round(2).sort_values("MAPE (%)")

,MAPE (%),MAE (juta),RMSE (juta),R2
XGBoost,24.16,1240.61,5537.09,0.71
LightGBM,24.34,1315.69,5636.07,0.71
Ridge (fitur miring di-log),24.60,1212.76,5339.61,0.74
Random Forest,24.84,1267.51,5471.15,0.72
Ridge (fitur mentah),47.21,13323.39,251398.33,-1663.83
Baseline: tebak median,91.30,3230.62,10361.29,-0.07


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from scipy.stats import randint, uniform, loguniform

def mape_rupiah(y_log, pred_log):
    return mean_absolute_percentage_error(np.expm1(y_log), np.expm1(pred_log)) * 100

skor_mape = make_scorer(mape_rupiah, greater_is_better=False)

ruang_cari = {
    "XGBoost": (XGBRegressor(random_state=42, n_jobs=-1), {
        "n_estimators":     randint(200, 1000),
        "learning_rate":    loguniform(0.01, 0.2),
        "max_depth":        randint(3, 9),
        "min_child_weight": randint(1, 10),
        "subsample":        uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.5, 0.5),
        "reg_lambda":       loguniform(0.1, 10),
    }),
    "LightGBM": (LGBMRegressor(random_state=42, verbose=-1), {
        "n_estimators":      randint(200, 1000),
        "learning_rate":     loguniform(0.01, 0.2),
        "num_leaves":        randint(8, 64),
        "min_child_samples": randint(5, 40),
        "subsample":         uniform(0.6, 0.4),
        "subsample_freq":    [1],
        "colsample_bytree":  uniform(0.5, 0.5),
        "reg_lambda":        loguniform(0.1, 10),
    }),
}

hasil_tuning = {}
for nama, (model, param) in ruang_cari.items():
    cari = RandomizedSearchCV(model, param, n_iter=25, scoring=skor_mape,
                              cv=gkf, random_state=42, n_jobs=1)
    cari.fit(X_train, y_train_log, groups=grup_train)
    hasil_tuning[nama] = cari
    print(nama, "| MAPE terbaik:", round(-cari.best_score_, 2))
    print("   setelan:", cari.best_params_)

XGBoost | MAPE terbaik: 21.91
   setelan: {'colsample_bytree': np.float64(0.5499874579090014), 'learning_rate': np.float64(0.03958189513876493), 'max_depth': 7, 'min_child_weight': 4, 'n_estimators': 863, 'reg_lambda': np.float64(2.0034427927560734), 'subsample': np.float64(0.6225646316108401)}
LightGBM | MAPE terbaik: 22.22
   setelan: {'colsample_bytree': np.float64(0.8091930046665436), 'learning_rate': np.float64(0.031448108040457876), 'min_child_samples': 7, 'n_estimators': 684, 'num_leaves': 58, 'reg_lambda': np.float64(2.2941144328643994), 'subsample': np.float64(0.7801997007878172), 'subsample_freq': 1}


In [7]:
for nama, cari in hasil_tuning.items():
    hasil[nama + " (tuning)"] = evaluasi_cv(cari.best_estimator_, X_train, y_train_log, grup_train)

pd.DataFrame(hasil).T.round(2).sort_values("MAPE (%)")

,MAPE (%),MAE (juta),RMSE (juta),R2
XGBoost (tuning),21.91,1149.42,5026.30,0.77
LightGBM (tuning),22.22,1178.82,4998.64,0.77
XGBoost,24.16,1240.61,5537.09,0.71
LightGBM,24.34,1315.69,5636.07,0.71
Ridge (fitur miring di-log),24.60,1212.76,5339.61,0.74
Random Forest,24.84,1267.51,5471.15,0.72
Ridge (fitur mentah),47.21,13323.39,251398.33,-1663.83
Baseline: tebak median,91.30,3230.62,10361.29,-0.07


In [ ]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test_log = pd.read_csv("../data/processed/y_test_log.csv").squeeze("columns")

model_final = clone(hasil_tuning["XGBoost"].best_estimator_)
model_final.fit(X_train, y_train_log)

pred = np.expm1(model_final.predict(X_test))
asli = np.expm1(y_test_log)
persen_meleset = np.abs(pred - asli) / asli * 100

print("MAPE (%)    :", round(mean_absolute_percentage_error(asli, pred) * 100, 2))
print("MAE (juta)  :", round(mean_absolute_error(asli, pred) / 1e6, 2))
print("RMSE (juta) :", round(root_mean_squared_error(asli, pred) / 1e6, 2))
print("R2          :", round(r2_score(asli, pred), 3))
print("Median meleset (%)       :", round(np.median(persen_meleset), 1))
print("Rumah meleset <= 20% (%) :", round((persen_meleset <= 20).mean() * 100, 1))

MAPE (%)    : 20.3
MAE (juta)  : 1141.4
RMSE (juta) : 4880.15
R2          : 0.6
Median meleset (%)       : 13.3
Rumah meleset <= 20% (%) : 65.9


In [9]:
import joblib
joblib.dump(model_final, "../models/model_xgb.joblib")

['../models/model_xgb.joblib']